# 4회차 · 첫 머신러닝 모델 — scikit-learn 으로 분류하기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KimKangSoo123/Github_Colab_Integration_Practice/blob/main/week4/04_first_ml.ipynb)

**오늘의 목표 (약 3시간)**
- 머신러닝이란? (학습 / 예측)
- 학습/테스트 데이터 분리 (train_test_split) 와 그 이유
- 과적합(overfitting) 개념
- scikit-learn 흐름: `모델 만들기 → fit → predict`
- 붓꽃 품종 분류 실습

**진행 방식**: 개념 → 예제 실행 → 🔧 빈칸 채우기 순서입니다.

*   과적합(Overfitting) 	: 기계학습 모델이 훈련 데이터에 너무 과하게 맞춰져서 실제 데이터나 새로운 데이터에 대한 예측 성능이 떨어지는 현상

*   scikit-learn(사이킷런) 	: 다섯 가지 주요 단계의 	데이터 분리, 전처리, 학습(fit), 예측(predict), 평가로 이루어진 핵심 작업 흐름

---
## 0. 머신러닝이란?

규칙을 사람이 일일이 짜는 대신, **데이터에서 패턴을 스스로 배우게** 하는 것입니다.
- **학습(fit)**: 정답이 있는 데이터를 보여주며 패턴을 익히게 함
- **예측(predict)**: 배운 패턴으로 새 데이터의 답을 맞힘

오늘은 붓꽃의 꽃잎/꽃받침 크기(특징 X)로 품종(정답 y)을 맞히는 **분류(classification)** 를 합니다.

In [6]:
import seaborn as sns
import pandas as pd

iris = sns.load_dataset("iris")

# X = 특징(입력), y = 정답(맞힐 대상)
X = iris[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = iris["species"]

print("X 모양:", X.shape)   # (150, 4)
print("y 종류:", y.unique())
X.head()

X 모양: (150, 4)
y 종류: ['setosa' 'versicolor' 'virginica']


,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


*   Seaborn : 예쁘고 복잡한 통계 그래프를 쉽게 만들 수 있게 도와주는 보조 도구 => 표

*   X = iris[["sepal_length", "sepal_width", "petal_length", "petal_width"]] : X라는 변수에 iris 데이터셋의 전체 150개 행은 그대로 유지하면서, 지정한 4개 열만 추출해 저장

    - 안쪽 []: 선택할 여러 열의 이름을 리스트로 묶음

*   .unique() : 데이터프레임 열에서 중복을 제거한 고유한 값들을 찾아 배열(array) 형태로 돌려주는 기능

---
## 1. 학습/테스트 분리 — 왜 나눌까?

배운 데이터로 시험 보면 당연히 잘 맞힙니다(그건 "외운 것"). 진짜 실력은 **처음 보는 데이터**로 재야 해요. 그래서 데이터를 학습용과 테스트용으로 나눕니다. 이렇게 안 하면 **과적합**(외우기만 하고 새 데이터엔 약한 상태)을 못 잡아냅니다.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# test_size=0.2 → 20%는 테스트용으로 떼어둠
# random_state=42 → 나누는 방식을 고정(재현 가능하게)

print("학습:", X_train.shape[0], "개 / 테스트:", X_test.shape[0], "개")

학습: 120 개 / 테스트: 30 개


*   train_test_split : 전체 데이터를 학습용 데이터(Train set)와 테스트용 데이터(Test set)로 나누어 주는 함수

*   X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) : X, y에 대해 X_train, X_test, y_train, y_test로 나눌 것이고 test_size는 전체 볼륨의 0.2에 해당하며 랜덤을 42로 설정하여 재현 가능성을 확보

    - 같은 행의 X와 y는 따로 무작위로 나뉘는 것이 아니라 서로 대응 관계를 유지한 채 함께 나뉨. (둘다 test에 들어가거나 train에 들어가거나)

*   X_train.shape[0] : X_train.shape은 (120, 4)로 [0]을 하게 되면 (120, 4)의 첫 번째 값인 120이 나오게 됨. (len()을 사용해도 자료의 총 길이인 120이 나옴.)

### 🔧 과제 1-1
`test_size` 를 `0.3` 으로 바꿔 다시 나누고, 학습/테스트 개수를 출력하세요.

In [9]:
# TODO
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
print("학습:", len(X_tr), "/ 테스트:", len(X_te))

학습: 105 / 테스트: 45


---
## 2. 모델 학습 & 예측 — 3단계

scikit-learn의 모든 모델은 똑같은 흐름입니다: **① 모델 생성 → ② fit(학습) → ③ predict(예측)**.
이해하기 쉬운 결정트리(Decision Tree)부터 써봅니다.

In [16]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1) 모델 생성
model = DecisionTreeClassifier(random_state=42)

# 2) 학습 — 학습용 데이터로만!
model.fit(X_train, y_train)

# 3) 예측 — 처음 보는 테스트 데이터로
pred = model.predict(X_test)

# 정확도: 맞힌 비율
acc = accuracy_score(y_test, pred)
print("테스트 정확도:", round(acc, 3))

테스트 정확도: 1.0


*   DecisionTreeClassifier 		: 데이터를 여러 갈래로 나누어 규칙을 찾는 분류 모델을 만드는 도구 -> 모델 생성

*   accuracy_score 			: 만든 모델이 얼마나 정확하게 맞혔는지 성능을 평가하는 도구 -> 정확도 평가

*   model.fit(X_train, y_train) 	: .fit()함수를 통해 "X_train, y_train"을 모델에 학습시켜 분류 규칙을 만듦.

*   pred = model.predict(X_test) 	: 학습이 끝난 모델에 "X_test"을 넣어서 나온 예측값을 "pred" 변수에 입력 (예측된 결과값이 저장됨)

*   accuracy_score(y_test, pred) 	: 정확도 = 맞힌 데이터 수(pred) ÷ 전체 테스트 데이터 수(y_test)

In [14]:
# 실제로 하나 예측해보기
sample = [[5.1, 3.5, 1.4, 0.2]]   # 어떤 붓꽃의 측정값
print("이 꽃의 예측 품종:", model.predict(sample)[0])

이 꽃의 예측 품종: setosa


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


*   /usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(

    : 모델을 학습시킬 때 사용한 데이터의 열 이름(특징 이름)과, 나중에 예측이나 평가에 넣은 데이터의 열 이름이 서로 다를 때 발생

    - 학습 시에는 판다스 데이터프레임(열 이름 있음)을 썼지만, 예측 시에는 넘파이 배열(numpy array)이나 이름이 없는 데이터를 넣었을 때 발생

### 🔧 과제 2-1
로지스틱 회귀(LogisticRegression)로 **같은 3단계**를 수행하고 정확도를 출력하세요.
알고리즘만 바뀌고 흐름은 완전히 똑같습니다.

In [19]:
from sklearn.linear_model import LogisticRegression

# 1) 모델 생성
logreg = LogisticRegression(max_iter=200)

# 2) 학습  TODO
logreg.fit(X_train, y_train)

# 3) 예측  TODO
pred_lr = logreg.predict(X_test)

print("로지스틱 회귀 정확도:", round(accuracy_score(y_test, pred_lr), 3))

로지스틱 회귀 정확도: 1.0


*   LogisticRegression 	: 사이킷런(scikit-learn) 라이브러리에서 제공하는 로지스틱 회귀(Logistic Regression) 알고리즘을 구현한 클래스로 데이터가 어떤 범주에 속할지 확률을 예측하여 분류(Classification) 문제를 풀 때 사용

*   max_iter=200 		: 모델이 정답을 찾기 위해 최적화 과정을 반복할 수 있는 최대 횟수를 200번으로 지정

---
## 3. 과적합 체감하기

학습 데이터 정확도와 테스트 정확도를 비교해봅니다. 학습 점수만 높고 테스트 점수가 낮으면 = 과적합 신호예요.

In [20]:
train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc = accuracy_score(y_test, model.predict(X_test))
print(f"학습 정확도: {train_acc:.3f}")
print(f"테스트 정확도: {test_acc:.3f}")
print("→ 두 값 차이가 크면 과적합을 의심합니다.")

학습 정확도: 1.000
테스트 정확도: 1.000
→ 두 값 차이가 크면 과적합을 의심합니다.


*   train_acc = accuracy_score(y_train, model.predict(X_train)) 	: 학습 정확도로 학습 결과값과 학습 데이터값으로 예측한 결과값으로 구함.


*   test_acc = accuracy_score(y_test, model.predict(X_test)) 	: 테스트 정확도로 테스트 결과값과 테스트 데이터값으로 예측한 결과값으로 구함.

*   f"문자열"			: 문자열 앞에 f(formatted string, 서식 문자열)를 붙이면 문자열 안의 {}에 변수나 계산식을 넣을 수 있다.


*   f"문자열{A:.2f}" 안에 :	: A(출력할 변수) 뒤에 출력 형식(:.3f - 소수점 3번째 자리로 지정)을 지정하겠다는 표시

### 🔧 오늘의 미니 과제 (제출용)
결정트리의 `max_depth`(트리 깊이)를 **1, 3, 없음(기본)** 으로 바꿔가며 테스트 정확도를 비교하세요. 깊이가 너무 깊으면 과적합되기 쉽습니다.

In [22]:
for depth in [1, 3, None]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)
    # TODO: 테스트 정확도를 구해 acc 에 저장
    pred_task = m.predict(X_test)
    acc_task = accuracy_score(y_test, pred_task)
    print(f"max_depth={depth}: 테스트 정확도 {acc_task:.3f}")

max_depth=1: 테스트 정확도 0.633
max_depth=3: 테스트 정확도 1.000
max_depth=None: 테스트 정확도 1.000


*   결정트리의 max_depth를 바꾸라는 것 : 트리가 질문을 최대 몇 번까지 이어가며 데이터를 나눌지 그 최대 허용 깊이를 정하라는 뜻

*   max_iter과 max_depth의 차이 : 둘다 모델 학습을 멈추게 하는 제어 도구이지만 반복 횟수 제한인지 나무의 깊이 제한인지 대상과 개념이 완전히 다름

    - max_iter은 경사 하강법 등을 쓰는 모델의 최대 반복 횟수 (로지스틱 회귀, MLP, SVM 등 반복 계산이 필요한 알고리즘.)
    - max_depth는 결정 트리가 뻗어나갈 수 있는 최대 층수 (결정 트리(Decision Tree) 및 이를 기반으로 한 앙상블 모델(랜덤 포레스트, XGBoost 등).)


**📝 어떤 깊이가 가장 좋았나요? 한 줄로 적어보세요:

max_depth=3, None의 테스트 정확도가 모두 1.000으로 가장 높았으나 동일한 정확도라면 구조가 더 단순하고 과적합 위험이 낮은 max_depth=3이 가장 적절하다고 판단하였다.

---
## 마무리 & 제출

**제출 방법**
1. `런타임 → 모두 실행` 으로 전체를 돌려 결과를 채웁니다.
2. `파일 → GitHub에 사본 저장` → 경로를 `members/본인이름/04_first_ml.ipynb` 로 지정합니다.
3. 커밋 메시지: `4회차 완료`

**복습 팁**: 막혔던 셀 아래에 마크다운 셀을 추가해 "왜 헷갈렸는지"를 한두 줄 적어두면 최고의 복습 노트가 됩니다.


**다음 회차 예고 (5회차)**: 정확도 하나만 보면 속을 수 있습니다. 정밀도·재현율·혼동행렬로 모델을 제대로 평가하는 법을 배웁니다.